# RAG Pipeline Flow 
This notebook walks through every stage of the RAG pipeline in order.
Each cell maps directly to one file in the `indexing/` or `generation/` folder.

**Pipeline:**
```
INDEXING                          GENERATION
────────────────────────────      ──────────────────────────
1. Load     sample_news.json      5. Embed query
2. Chunk    split into pieces     6. Retrieve  vector search
3. Embed    text → vectors        7. Generate  LLM + context
4. Store    vectors → Qdrant      8. Response  citations
```

In [1]:
import json, os 
from dotenv import load_dotenv
from datetime import datetime # parsing date time

load_dotenv()  # reads from .env

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
assert OPENAI_API_KEY, "Add OPENAI_API_KEY to your .env file"
print("✓ API key loaded")

✓ API key loaded


# Indexing Pipeline

## Stage 1: Load documents
`indexing/loaders/file_loader.py`

Read raw documents and outputs a list of `documents`

In [6]:
class RawDocument:
    def __init__(self, id, title, text, url, publish_date, category="", tags=None):
        self.id = id
        self.title = title
        self.text = text
        self.url = url
        self.publish_date = publish_date
        self.category = category
        self.tags = tags if tags is not None else []
        
def load(path):
    with open(path, encoding="utf-8") as f:
        raw = json.load(f)
        
    docs = []
    for doc in raw: 
        docs.append(RawDocument(
            id=doc["id"],
            title=doc["title"],
            text=doc["content"],
            url=doc["url"],
            publish_date=datetime.fromisoformat(doc["publish_date"]),
            category=doc.get("category", ""),
            tags=doc.get("tags", []),
        ))
        
    return docs
        
docs = load("./data/news.json")

print(f"Loaded {len(docs)} documents\n")
for doc in docs:
    print(f" [{doc.id}] {doc.title}")

Loaded 11 documents

 [20260203_1] 樂團「微醺開根」首發專輯 唱當代族群故事
 [20260203_2] 排灣童聲唱出部落記憶 春日國小推《春迴》專輯
 [20260203_3] 仁愛鄉梅子白蘭地風味絕佳 奪國際一星獎章
 [20260203_4] 復興推「部落社區再復興」 凝共識促永續發展
 [20260203_5] 陳義信用行動感謝 職棒生涯文物捐贈退輔會
 [20260203_6] 台北國際電玩展 設計商「赴原Saikin」展新作
 [20260203_7] 嘉義文化科技創意賽頒獎 近50組團隊參賽
 [20260203_8] 《村裡遇見熊》書展亮相 記錄人熊共生故事錄
 [20260203_9] 台北國際書展登場 原民館推講座、交流活動
 [20260203_11] 首批承領人正式取得 大南澳土地所有權狀
 [20260203_12] 冒用縣府名義詐捐 花蓮縣府急澄清籲防詐


# Stage 2: Chunk
`indexing/chunkers/semantic_chunker.py`

Split each document into smaller pieces. 

**Why**: LLMs have content limits. Smaller chunks = more precise retrieval

Key rule: every chunk must keep `title`, `url`, `publish_date` for citation later.

In [17]:
import re

class Chunk:
    def __init__(self, chunk_id, source_id, text, title, url, publish_date):
        self.chunk_id = chunk_id
        self.source_id = source_id
        self.text = text
        self.title = title
        self.url = url
        self.publish_date = publish_date
        
def chunk(doc: RawDocument, chunk_size=200, overlap=100):
    """
    Use sentence-aware sliding window chunking for the baseline
    """
    # Split on Chinese punctuation
    sentences = re.split(r"(?<=[。！？])", doc.text)
    sentences = [s.strip() for s in sentences if s.strip()]
    
    chunks, current, current_len = [], [], 0
    for sent in sentences:
        if current_len + len(sent) > chunk_size and current:
            chunks.append("".join(current))
            current = current[-overlap:] if overlap else []
            current_len = sum(len(s) for s in current)
        current.append(sent)
        current_len += len(sent)
    if current:
        chunks.append("".join(current))
        
    return [
        Chunk(
            source_id=doc.id,
            chunk_id=f"{doc.id}_{i}",
            text=text,
            title=doc.title,
            url=doc.url,
            publish_date=doc.publish_date,
        )
        for i, text in enumerate(chunks)
    ]

all_chunks = []
for doc in docs:
    doc_chunk = chunk(doc)
    all_chunks.extend(doc_chunk)
    
print(f"{len(docs)} documents → {len(all_chunks)} chunks\n")

11 documents → 80 chunks



In [22]:
# Inspect one document's chunk.
example_chunks = [c for c in all_chunks if c.source_id == "20260203_1"]
print(f"This news(id = 20260203_1) is split in to {len(example_chunks)} chunks") # These chunks are all belong to the news (id=20260203_1)
for c in example_chunks:
    print(f"\n [{c.chunk_id}] ({len(c.text)}) chars")
    print(f" {c.text[80:]}...")

This news(id = 20260203_1) is split in to 8 chunks

 [20260203_1_0] (194) chars
 軸。微醺開根團長 Teymu Boya（以樂） ：「這首歌其實就在講述樂團從一剛開始，然後到成軍、到現在，一路走過來，無論好的、壞的，我們都用比喻的方式，寫成像是在生活的過程，可能會遇到挫折、遇到貴人這些，我們都寫在這首歌裡面。...

 [20260203_1_1] (253) chars
 軸。微醺開根團長 Teymu Boya（以樂） ：「這首歌其實就在講述樂團從一剛開始，然後到成軍、到現在，一路走過來，無論好的、壞的，我們都用比喻的方式，寫成像是在生活的過程，可能會遇到挫折、遇到貴人這些，我們都寫在這首歌裡面。」
團長Teymu說，團員就像圍著火堆取暖，在音樂中逐漸凝聚彼此，也透過創作，開啟年輕世代與原鄉、身分認同之間的對話。...

 [20260203_1_2] (299) chars
 軸。微醺開根團長 Teymu Boya（以樂） ：「這首歌其實就在講述樂團從一剛開始，然後到成軍、到現在，一路走過來，無論好的、壞的，我們都用比喻的方式，寫成像是在生活的過程，可能會遇到挫折、遇到貴人這些，我們都寫在這首歌裡面。」
團長Teymu說，團員就像圍著火堆取暖，在音樂中逐漸凝聚彼此，也透過創作，開啟年輕世代與原鄉、身分認同之間的對話。分享會上，樂團也發表一首為平埔族群創作的歌曲，希望讓更多人透過音樂，理解不同族群的文化脈絡。...

 [20260203_1_3] (349) chars
 軸。微醺開根團長 Teymu Boya（以樂） ：「這首歌其實就在講述樂團從一剛開始，然後到成軍、到現在，一路走過來，無論好的、壞的，我們都用比喻的方式，寫成像是在生活的過程，可能會遇到挫折、遇到貴人這些，我們都寫在這首歌裡面。」
團長Teymu說，團員就像圍著火堆取暖，在音樂中逐漸凝聚彼此，也透過創作，開啟年輕世代與原鄉、身分認同之間的對話。分享會上，樂團也發表一首為平埔族群創作的歌曲，希望讓更多人透過音樂，理解不同族群的文化脈絡。微醺開根成員 柳町 & 粉絲 Tabiliah ：「就可以去凝聚更多的力量，讓更多人知道、認識他們。...

 [20260203_1_4] (467) chars
 軸。微醺開根團長

# Stage 3: Embed
`indexing/embedders/bge_m3.py`

Convert each chunk's text into a vector representation. *Why*: vectors capture semantic meaning and similar meaning means similar vector.

In [ ]:
from sentence_transformers import SentenceTransformer

# Embedder Choice: "BAAI/bge-m3" / ""
embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

texts = [chunk.text for chunk in all_chunks]
embeddings = embedder.encode(texts, normalize_embeddings=True, show_progress_bar=True)

print(f"\nEmbedded {len(embeddings)} chunks")
print(f"Vector dimension: {len(embeddings[0])}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4153.60it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 3/3 [00:00<00:00, 43.82it/s]


Embedded 80 chunks
Vector dimension: 384


# Stage 4: Store
`indexing/vector_stores/qdrant_store.py`

Save vectors + chunks metadata into Qdrant.

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct 

# :memory: = no Docker needed. Swap to host="localhost" when using Docker
client = QdrantClient(":memory:")
